# BITS Academic Course Recommender — FINAL SUBMISSION

**Postman Round 2 · Tasks 1–7, 9 and 10 only.** Optional Task 8 timetable intelligence is intentionally excluded.

This notebook launches a working, source-grounded dashboard with editable profiles, academic requirement analysis, deterministic course gates, natural-language queries, evidence citation, and optional LLM intent understanding. It reads the **provided** BITS Bulletin 2025–26, Academic Regulations 2023, Pilani timetable 2026–27, and **540 handout PDFs** via the bundled repository archive.

**Required download/upload:** `BITS_Course_Recommender_Repository.zip` (downloaded alongside this notebook). Run all cells from top to bottom. The archived structured dataset enables a fast start; raw source PDFs and the parser are included for reproducibility.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, zipfile, importlib.util
IS_COLAB = 'google.colab' in sys.modules
WORKSPACE = Path('/content') if IS_COLAB else Path.cwd()
WORKSPACE.mkdir(parents=True, exist_ok=True)
print('Running in:', 'Google Colab' if IS_COLAB else 'local Python')
print('Workspace:', WORKSPACE)

In [ ]:
# Install only if a runtime is missing one of the required libraries.
required = [('fitz','PyMuPDF>=1.24,<2'),('gradio','gradio>=4.44,<7'),('pytest','pytest>=8,<10')]
missing = [package for module,package in required if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
else:
    print('Required Python packages already available.')

In [ ]:
# ONE upload: repository archive, already containing all 4 original source files.
ARCHIVE_NAME = 'BITS_Course_Recommender_Repository.zip'
archive = Path(os.environ.get('BITS_REPO_ZIP', str(WORKSPACE / ARCHIVE_NAME)))
folder = WORKSPACE / 'BITS_Course_Recommender_Submission'
if not folder.joinpath('app.py').exists():
    if not archive.is_file():
        if IS_COLAB:
            from google.colab import files
            print('Upload the supplied', ARCHIVE_NAME, '(large upload; please wait)')
            uploaded=files.upload()
            if ARCHIVE_NAME not in uploaded:
                raise FileNotFoundError('Expected '+ARCHIVE_NAME+'. Do not rename the supplied archive.')
            archive=WORKSPACE/ARCHIVE_NAME
        else:
            raise FileNotFoundError('Place '+ARCHIVE_NAME+' beside the notebook, or set BITS_REPO_ZIP to its absolute path.')
    with zipfile.ZipFile(archive) as bundle:
        # Reject dangerous paths when unpacking an untrusted ZIP.
        for item in bundle.infolist():
            target=(WORKSPACE/item.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise ValueError('Unsafe ZIP member: '+item.filename)
        bundle.extractall(WORKSPACE)
assert folder.joinpath('app.py').exists(), 'Repository extraction failed'
os.chdir(folder)
sys.path.insert(0,str(folder))
print('Ready:',folder)

In [ ]:
# Verify that the original inputs have not been replaced by a toy dataset.
source_names=['bulletin.pdf','regulations.pdf','timetable.pdf','handouts.zip']
for item in source_names:
    path=folder/'sources'/item
    assert path.is_file() and path.stat().st_size>1000, 'Missing: '+str(path)
    print(f'{item:20} {path.stat().st_size / (1024**2):8.1f} MiB')
with zipfile.ZipFile(folder/'sources'/'handouts.zip') as original:
    handout_count=sum(n.filename.lower().endswith('.pdf') for n in original.infolist())
assert handout_count == 540, f'Expected 540 handouts but received {handout_count}'
print('PASS: original handout ZIP has',handout_count,'PDFs.')

In [ ]:
# The ingestion pipeline can be rerun for a new semester. Use shipped JSON for startup.
from src.ingest import build_dataset
DATA_PATH=folder/'data'/'academic_dataset.json'
REBUILD_FROM_ORIGINAL_PDFS=False
if REBUILD_FROM_ORIGINAL_PDFS or not DATA_PATH.exists():
    build_dataset(folder/'sources',DATA_PATH)
from src.engine import load_data
db=load_data(DATA_PATH)
assert db['ingestion_report']['pdf_files_seen']==540
assert db['ingestion_report']['offered_course_codes']>250
assert len(db['programmes'])>=15
print('PASS: structured dataset loaded.')
print('Records:',len(db['courses']),'Offered codes:',db['ingestion_report']['offered_course_codes'])
print('Handout-linked codes:',db['ingestion_report']['handout_course_codes'])

In [ ]:
# Review the provenance for a REAL source-extracted course.
sample=db['courses']['CS F407']
print('Course:',sample['code'],sample['title'])
print('Timetable evidence:',sample['offers'][0]['source'])
print('Degree classification:',sample['categories'].get('B.E. Computer Science'))
print('Handout:',sample['handouts'][0]['source'])
print('Midsem presence:',sample['handouts'][0]['midsem'])
assert sample['handouts'][0]['source']['document'].lower().endswith('.pdf')

In [ ]:
# Example profile: edit freely in the launched dashboard.
from src.engine import profile_from_inputs,applicable_requirements,format_requirements
example_profile=profile_from_inputs(
   campus='Pilani',admission_year=2025,primary='B.E. Electrical and Electronics',
   secondary='M.Sc. Semiconductor and Nanoscience',semester=3,
   completed='MATH F101, PHY F101',current='',minor='',interests='AI and semiconductor devices')
state=applicable_requirements(db,example_profile)
assert 'requirements' in state
print(format_requirements(db,example_profile))

In [ ]:
# Live query path: never hardcoded recommendations.
from src.engine import query_courses,format_recommendations
for question in ['Suggest DELs related to AI',
                 'I want a HUEL with project-based evaluation',
                 'Find an OPEL with no mandatory attendance']:
    out=query_courses(db,example_profile,question,limit=4)
    print('\nQUESTION:',question)
    print('Matching candidates:',out.get('candidate_count',0))
    for item in out.get('results',[]):
        print('  ',item['code'],item['category'],'|',item['title'],'| PROVISIONAL')
    if not out.get('results'):print('  No explicitly verified matches; no properties invented.')

In [ ]:
# Automated regression suite. This verifies eligibility gates, documented policy, callbacks, and UI construction.
run=subprocess.run([sys.executable,'-m','pytest','-q','tests'],cwd=folder,
                   capture_output=True,text=True,timeout=180)
print(run.stdout[-6000:])
if run.returncode:
    print(run.stderr[-4000:])
assert run.returncode==0,'Tests failed; do NOT submit until corrected.'
print('PASS: all repository regression tests passed.')

In [ ]:
# Build the actual Gradio dashboard (not a static screenshot).
from app import create_app
dashboard=create_app()
assert len(dashboard.blocks)>15
print('PASS: dashboard UI created with',len(dashboard.blocks),'Gradio blocks.')

In [ ]:
# Optional: Colab Secrets → OPENAI_API_KEY. LLM is not required to run.
# Even with a key, only the query's intent goes to the API. Eligibility remains deterministic.
if IS_COLAB and not os.environ.get('OPENAI_API_KEY'):
    try:
        from google.colab import userdata
        secret=userdata.get('OPENAI_API_KEY')
        if secret: os.environ['OPENAI_API_KEY']=secret
    except Exception:
        pass
print('Optional API intent mode:', 'configured' if os.environ.get('OPENAI_API_KEY') else 'off (local parser works)')

In [ ]:
# TASK 10 WORKING DASHBOARD. Open the Gradio URL printed below.
# CI_MODE=1 skips public launch during automated notebook validation.
if os.environ.get('CI_MODE')=='1':
    print('CI mode: external Gradio sharing skipped; dashboard constructed and unit-tested above.')
else:
    dashboard.launch(share=IS_COLAB,inline=False,show_error=True,
                     prevent_thread_lock=True)

## Repository / submission instructions

The **second mandatory deliverable** is a clean Git repository. The provided ZIP contains `app.py`, `src/ingest.py`, `src/engine.py`, `src/llm.py`, `tests/`, structured data and `README.md`. Publish this code to your own GitHub repository using the README instructions; do **not** commit the ~217 MB raw handout ZIP (GitHub size limit). The source archive is kept in the download package for reproducibility.

**Honest limitations:** Prerequisite/restriction details are *not* fully supplied (timetable p.116 points outside); the app flags final eligibility as unverified. Completed-course unit totals missing from available timetable records are marked unverified. No timetable-section or exam-clash logic (optional Task 8) is implemented. The live Gradio public URL requires Google Colab network access.